In [1]:
from transformers import AutoTokenizer
from transformers import AutoConfig
import torch
from src.model import XMistralForCausalLM
import random

In [2]:
model_name_or_path = 'Hannibal046/xrag-7b'
device = "cuda"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path,
    padding_side = 'left',
    add_eos_token=False, ## import to include this!
    use_fast=False,
)
if tokenizer.pad_token:
    pass
elif tokenizer.unk_token:
    tokenizer.pad_token_id = tokenizer.unk_token_id
elif tokenizer.eos_token:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [4]:
config = AutoConfig.from_pretrained(model_name_or_path)
MODEL_CLASS = eval(config.architectures[0])
model = MODEL_CLASS.from_pretrained(
    model_name_or_path,
    torch_dtype = torch.bfloat16,
    low_cpu_mem_usage = True,
    device_map= device,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
XRAG_TOKEN = "<xRAG>" 

In [7]:
ParaphraseInstructions = [
    'Background: {xrag_token} means the same as',
    "Background: {xrag_token} Can you put the above sentences in your own terms?",
    "Background: {xrag_token} Please provide a reinterpretation of the preceding background text.",
    "These two expressions are equivalent in essence:\n(1) {xrag_token}\n(2)",
    "Background: {xrag_token} is a paraphrase of what?",
    "Background: {xrag_token} Could you give me a different version of the background sentences above?",
    "In other words, background: {xrag_token} is just another way of saying:",
    "You're getting across the same point whether you say background: {xrag_token} or",
    "Background: {xrag_token} After uppacking the ideas in the background information above, we got:",
    "Background: {xrag_token} Please offer a restatement of the background sentences I've just read.",
    "Background: {xrag_token}, which also means:",
    "Strip away the mystery, and you'll find background: {xrag_token} is simply another rendition of:",
    "The essence of background: {xrag_token} is captured again in the following statement:",
]

In [8]:
prmopts2 = [
    random.choice(ParaphraseInstructions).format_map(dict(xrag_token=XRAG_TOKEN)) for _ in range(3)
]

In [9]:
prmopts2

['Background: <xRAG> Could you give me a different version of the background sentences above?',
 'Background: <xRAG>, which also means:',
 'Background: <xRAG> Can you put the above sentences in your own terms?']

In [10]:
all_messages = [
    [{"role": "user", "content": p} 
] for p in prmopts2]

In [11]:
applide_prompts = tokenizer.apply_chat_template(all_messages, tokenize=False)

In [12]:
from src.eval.run_eval import llm_for_open_generation

In [13]:
from src.eval.symantic.embedding_utils import get_documents_embeds

In [14]:
r = get_documents_embeds('brimmann2/squad_qa1', "train")

loading embedder model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

embedder model loaded
generating embeddings...
embeddings generated


In [19]:
len(r)

3

In [16]:
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))

In [17]:
results = llm_for_open_generation(model,tokenizer,applide_prompts,r, 4, True)

4it [00:16,  4.15s/it]


In [18]:
results

["In January 2009, President Barack Obama announced the United States' commitment to the United Nations Population Fund (UNFPA), reversing the Bush Administration's decision to cut off funding to the organization. The United States is the largest donor to the UNFPA, and the funding cut had been a major source of controversy. The Obama Administration's decision was welcomed by the international community, including the United Nations, which had called for the funding to be restored.",
 "After the Sex Pistols disbanded, Lydon formed Public Image Ltd. with Keith Levene, John Lydon's former bandmate in the Sex Pistols, and Jah Wobble, a bassist who had previously played with the band The Slits. The band's music was a departure from the punk sound of the Sex Pistols, incorporating elements of dub, funk, and avant-garde music. The band's first single",
 'The mosaic floor of the Peristyle of the Four Seasons in the House of the Vettii, Pompeii, is a famous example of the mosaic art of the Rom

In [50]:
applide_prompts

['<s>[INST] Background: <xRAG>, which also means: [/INST]',
 "<s>[INST] Strip away the mystery, and you'll find background: <xRAG> is simply another rendition of: [/INST]",
 '<s>[INST] Background: <xRAG>, which also means: [/INST]',
 '<s>[INST] Background: <xRAG>, which also means: [/INST]',
 "<s>[INST] You're getting across the same point whether you say background: <xRAG> or [/INST]",
 "<s>[INST] You're getting across the same point whether you say background: <xRAG> or [/INST]",
 '<s>[INST] The essence of background: <xRAG> is captured again in the following statement: [/INST]',
 '<s>[INST] Background: <xRAG> means the same as [/INST]',
 '<s>[INST] The essence of background: <xRAG> is captured again in the following statement: [/INST]',
 '<s>[INST] Background: <xRAG>, which also means: [/INST]',
 '<s>[INST] Background: <xRAG> After uppacking the ideas in the background information above, we got: [/INST]',
 '<s>[INST] Background: <xRAG> After uppacking the ideas in the background inf

In [19]:
messages = [
    {"role": "user", "content": instruction},
]

In [ ]:
applied_prompts = tokenizer.apply_chat_template(messages, tokenize=False)

In [23]:
encodeds

'<s>[INST] Background: <xRAG> is a paraphrase of what? [/INST]'

In [ ]:
encodeds = tokenizer.apply_chat_template(messages, tokenize=False)

In [24]:
assert XRAG_TOKEN in tokenizer.get_vocab()

In [25]:
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))

In [26]:
from src.eval.run_eval import prepare_prompts

In [27]:
def transform_messages(example, idx):
        """
        Extracts user question and assistant answer from messages,
        and assigns a new integer ID.
        """
        question = ""
        answer = ""
        for message in example['messages']:
            if message['role'] == 'user':
                question = message['content'].strip()
            elif message['role'] == 'assistant':
                answer = message['content'].strip()
                
        return {
            'id': idx,
            'question': question,
            'answer': answer,
            'background': [example['background']]
        }

In [29]:
from datasets import load_dataset

In [31]:
ds = load_dataset("brimmann2/squad-xgemma3-1")
ds1 = ds.map(transform_messages, with_indices=True, remove_columns=['id', 'messages', 'task_type'])
test_data = list(ds1["train"].select(range(3)))
dev_data = None

In [35]:
def create_prompt_with_mistral_chat_format(messages,tokenizer,*args,**kwargs):
    # return tokenizer.apply_chat_template(messages,tokenize=False,add_special_tokens=False)
    formatted_text = ""
    for message in messages:
        if message['role'] == 'user':
            formatted_text += "[INST] " + message['content'] + " [/INST]"
        elif message['role'] == 'assistant':
            formatted_text += message['content'] + tokenizer.eos_token
        else:
            raise ValueError(
                "Mistral chat template only supports 'user' and 'assistant' roles. Invalid role: {}.".format(message["role"])
                )
    # formatted_text += " The answer is:"
    return formatted_text

In [36]:
chat_format = create_prompt_with_mistral_chat_format

In [37]:
prompts, backgrounds = prepare_prompts(dev_data=dev_data, test_data=test_data, task_type = "open_qa", tokenizer = tokenizer, n_shot=0, use_rag=True, retrieval_embed_length=1, chat_format=chat_format)

**************************************** show one example ****************************************
[INST] Refer to the background document and answer the questions:

Background: <xRAG>

Question: Can you tell me the answer to What program funded in 2009 does not help women??? [/INST] The answer is:
**************************************** show one example ****************************************


In [38]:
prompts

['[INST] Refer to the background document and answer the questions:\n\nBackground: <xRAG>\n\nQuestion: Can you tell me the answer to What program funded in 2009 does not help women??? [/INST] The answer is:',
 '[INST] Refer to the background document and answer the questions:\n\nBackground: <xRAG>\n\nQuestion: What group declared itself to be anti-music of any kind????? [/INST] The answer is:',
 '[INST] Refer to the background document and answer the questions:\n\nBackground: <xRAG>\n\nQuestion: Please answer this question: The Beauty of Durres is in what country?? [/INST] The answer is:']

In [39]:
_ = model.eval()